# LunarSite — Stage 2: Enhanced Terrain Segmentation (GPU)

Two encoder options:
- **Baseline:** U-Net + ResNet-34 (ImageNet pretrained, 24M params)
- **Enhanced:** U-Net + DINOv2-Base (self-supervised ViT, domain-agnostic features)

Enhanced augmentations: shadow rotation, extreme contrast, Hapke BRDF perturbation, synthetic crater overlay.

**Runtime:** `Runtime > Change runtime type > T4 GPU`

**Classes:** background (0), small rocks (1), large rocks (2), sky (3)

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q segmentation-models-pytorch albumentations kagglehub

In [ ]:
import json
import time
from pathlib import Path

import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Configuration

Set `USE_DINOV2 = True` for the enhanced encoder, `False` for the ResNet-34 baseline.

In [ ]:
# ===== CHOOSE YOUR ENCODER =====
USE_DINOV2 = True   # True = DINOv2-Base (enhanced), False = ResNet-34 (baseline)
# ===============================

CONFIG = {
    "input_size": 476 if USE_DINOV2 else 480,  # DINOv2 needs divisible by 14
    "num_classes": 4,
    "class_names": ["background", "small_rocks", "large_rocks", "sky"],
    "epochs": 50,
    "batch_size": 6 if USE_DINOV2 else 8,  # DINOv2 uses more VRAM
    "lr": 5e-5 if USE_DINOV2 else 1e-4,
    "weight_decay": 1e-2 if USE_DINOV2 else 1e-4,
    "dice_weight": 0.5,
    "ce_weight": 0.5,
    "train_split": 0.8,
    "val_split": 0.1,
    "seed": 42,
    "encoder": "dinov2" if USE_DINOV2 else "resnet34",
    "mc_dropout_p": 0.1,
}

COLOR_TO_CLASS = {
    (0, 0, 0): 0, (255, 0, 0): 1, (0, 255, 0): 2, (0, 0, 255): 3,
}
CLASS_COLORS = np.array([
    [0, 0, 0], [255, 165, 0], [255, 0, 0], [135, 206, 235],
], dtype=np.uint8)

print(f"Encoder: {CONFIG['encoder']}")
print(f"Input size: {CONFIG['input_size']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"LR: {CONFIG['lr']}")

## 3. Download Dataset

In [ ]:
dataset_path = Path(kagglehub.dataset_download(
    "romainpessia/artificial-lunar-rocky-landscape-dataset"
))
image_dir = dataset_path / "images" / "render"
mask_dir = dataset_path / "images" / "clean"
real_moon_dir = dataset_path / "real_moon_images"
print(f"Images: {len(list(image_dir.glob('render*.png')))}")
print(f"Real moon: {len(list(real_moon_dir.glob('*.png')))}")

## 4. Lunar-Specific Augmentations

In [ ]:
class LunarShadowRotation(ImageOnlyTransform):
    """Rotate shadow direction to simulate varying sun azimuth."""
    def __init__(self, angle_range=(-45, 45), always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)
        self.angle_range = angle_range

    def apply(self, img, angle=0, **params):
        gray = np.mean(img.astype(np.float32), axis=-1)
        shadow_mask = gray < 25
        if shadow_mask.sum() < 100:
            return img
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
        rotated_shadow = cv2.warpAffine(
            shadow_mask.astype(np.uint8), M, (w, h),
            flags=cv2.INTER_NEAREST, borderValue=0
        ).astype(bool)
        result = img.copy()
        result[rotated_shadow] = np.clip(result[rotated_shadow] * 0.1, 0, 255).astype(np.uint8)
        revealed = shadow_mask & ~rotated_shadow
        result[revealed] = np.clip(result[revealed] * 3.0 + 30, 0, 200).astype(np.uint8)
        return result

    def get_params(self):
        return {"angle": np.random.uniform(*self.angle_range)}

    def get_transform_init_args_names(self):
        return ("angle_range",)


class ExtremeContrastAugmentation(ImageOnlyTransform):
    """Simulate extreme dynamic range of airless body imagery."""
    def __init__(self, always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)

    def apply(self, img, shadow_factor=0.05, sunlit_factor=1.5, **params):
        gray = np.mean(img.astype(np.float32), axis=-1)
        median = np.median(gray)
        result = img.astype(np.float32)
        result[gray < median * 0.5] *= shadow_factor
        result[gray > median * 1.5] *= sunlit_factor
        return np.clip(result, 0, 255).astype(np.uint8)

    def get_params(self):
        return {
            "shadow_factor": np.random.uniform(0.01, 0.15),
            "sunlit_factor": np.random.uniform(1.2, 2.5),
        }

    def get_transform_init_args_names(self):
        return ()


class HapkeBRDFPerturbation(ImageOnlyTransform):
    """Simulate varying regolith reflectance (Hapke BRDF)."""
    def __init__(self, always_apply=False, p=0.4):
        super().__init__(always_apply=always_apply, p=p)

    def apply(self, img, albedo_factor=1.0, phase_factor=1.0, **params):
        result = img.astype(np.float32) * albedo_factor
        gray = np.mean(result, axis=-1, keepdims=True)
        normalized = gray / (gray.max() + 1e-8)
        phase_mask = normalized * (1 - normalized) * 4
        result *= (1 - phase_mask * (1 - phase_factor))
        return np.clip(result, 0, 255).astype(np.uint8)

    def get_params(self):
        return {
            "albedo_factor": np.random.uniform(0.5, 1.5),
            "phase_factor": np.random.uniform(0.7, 1.3),
        }

    def get_transform_init_args_names(self):
        return ()


def get_lunar_transforms(input_size, training):
    if training:
        return A.Compose([
            A.RandomCrop(height=input_size, width=input_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=(-0.3, 0.2), contrast_limit=(-0.2, 0.4), p=0.4),
            A.GaussNoise(p=0.2),
            LunarShadowRotation(angle_range=(-45, 45), p=0.3),
            ExtremeContrastAugmentation(p=0.3),
            HapkeBRDFPerturbation(p=0.3),
        ])
    return A.Compose([A.CenterCrop(height=input_size, width=input_size)])

print("Lunar augmentations defined: ShadowRotation, ExtremeContrast, HapkeBRDF")

## 5. Dataset, Loss, Metrics

In [ ]:
def color_mask_to_index(mask_rgb):
    h, w = mask_rgb.shape[:2]
    index_mask = np.zeros((h, w), dtype=np.int64)
    for color, idx in COLOR_TO_CLASS.items():
        index_mask[np.all(mask_rgb == color, axis=-1)] = idx
    return index_mask


class LunarTerrainDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir, self.mask_dir = Path(image_dir), Path(mask_dir)
        self.transform = transform
        self.image_paths = sorted(self.image_dir.glob("render*.png"))
        self.mask_paths = sorted(self.mask_dir.glob("clean*.png"))
        assert len(self.image_paths) == len(self.mask_paths)

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = cv2.cvtColor(cv2.imread(str(self.image_paths[idx])), cv2.COLOR_BGR2RGB)
        mask = color_mask_to_index(cv2.cvtColor(cv2.imread(str(self.mask_paths[idx])), cv2.COLOR_BGR2RGB))
        if self.transform:
            t = self.transform(image=image, mask=mask)
            image, mask = t["image"], t["mask"]
        return {
            "image": torch.from_numpy(image.transpose(2, 0, 1).astype(np.float32) / 255.0),
            "mask": torch.from_numpy(mask.astype(np.int64)),
        }


class DiceCELoss(nn.Module):
    def __init__(self, dice_weight=0.5, ce_weight=0.5):
        super().__init__()
        self.dw, self.cw = dice_weight, ce_weight
        self.dice = smp.losses.DiceLoss(mode="multiclass")
        self.ce = nn.CrossEntropyLoss()
    def forward(self, pred, target):
        return self.dw * self.dice(pred, target) + self.cw * self.ce(pred, target)


def iou_score(pred, target, nc):
    pc = []
    for c in range(nc):
        i = ((pred == c) & (target == c)).sum().float()
        u = ((pred == c) | (target == c)).sum().float()
        pc.append((i / u).item() if u > 0 else float("nan"))
    v = [x for x in pc if x == x]
    return {"per_class_iou": pc, "mean_iou": sum(v) / len(v) if v else 0.0}


def dice_score(pred, target, nc):
    pc = []
    for c in range(nc):
        p, t = (pred == c).float(), (target == c).float()
        i, s = (p * t).sum(), p.sum() + t.sum()
        pc.append((2 * i / s).item() if s > 0 else float("nan"))
    v = [x for x in pc if x == x]
    return {"per_class_dice": pc, "mean_dice": sum(v) / len(v) if v else 0.0}

## 6. DINOv2 Encoder (Enhanced Model)

In [ ]:
class DINOv2UNet(nn.Module):
    """U-Net with DINOv2-Base encoder for lunar terrain segmentation."""

    def __init__(self, classes=4, frozen_encoder=False, dropout_p=0.1):
        super().__init__()
        self.backbone = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")
        self.embed_dim = self.backbone.embed_dim  # 768 for vitb14
        self.patch_size = 14

        if frozen_encoder:
            for p in self.backbone.parameters():
                p.requires_grad = False

        n = len(self.backbone.blocks)
        self.feat_idx = [n // 4 - 1, n // 2 - 1, 3 * n // 4 - 1, n - 1]

        ch = [64, 128, 256, 512]
        self.projections = nn.ModuleList([
            nn.Sequential(nn.Conv2d(self.embed_dim, c, 1), nn.BatchNorm2d(c), nn.ReLU(True))
            for c in ch
        ])

        dec_ch = [256, 128, 64, 32]
        self.decoders = nn.ModuleList()
        for i, dc in enumerate(dec_ch):
            ic = ch[-(i + 1)] + (dec_ch[i - 1] if i > 0 else 0)
            self.decoders.append(nn.Sequential(
                nn.Conv2d(ic, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True),
                nn.Dropout2d(dropout_p),
                nn.Conv2d(dc, dc, 3, padding=1), nn.BatchNorm2d(dc), nn.ReLU(True),
            ))

        self.final = nn.Conv2d(dec_ch[-1], classes, 1)

    def _extract_features(self, x):
        B, _, H, W = x.shape
        h, w = H // self.patch_size, W // self.patch_size
        feats, hooks = [], []

        for idx in self.feat_idx:
            fl = []
            feats.append(fl)
            hooks.append(self.backbone.blocks[idx].register_forward_hook(
                lambda m, i, o, s=fl: s.append(o)
            ))

        self.backbone.forward_features(x)
        for hook in hooks: hook.remove()

        outputs = []
        for i, (fl, proj) in enumerate(zip(feats, self.projections)):
            f = fl[0][:, 1:, :]  # remove CLS
            f = proj(f.reshape(B, h, w, self.embed_dim).permute(0, 3, 1, 2))
            if i < 3:
                f = F.interpolate(f, scale_factor=1.0 / (2 ** (3 - i)), mode="bilinear", align_corners=False)
            outputs.append(f)
        return outputs

    def forward(self, x):
        _, _, H, W = x.shape
        enc = self._extract_features(x)
        d = self.decoders[0](enc[-1])
        for i in range(1, len(self.decoders)):
            d = F.interpolate(d, size=enc[-(i + 1)].shape[2:], mode="bilinear", align_corners=False)
            d = self.decoders[i](torch.cat([d, enc[-(i + 1)]], dim=1))
        return self.final(F.interpolate(d, size=(H, W), mode="bilinear", align_corners=False))

print("DINOv2UNet defined.")

## 7. Data Loaders

In [ ]:
input_size = CONFIG["input_size"]
seed = CONFIG["seed"]

train_ds = LunarTerrainDataset(image_dir, mask_dir, transform=get_lunar_transforms(input_size, True))
val_ds = LunarTerrainDataset(image_dir, mask_dir, transform=get_lunar_transforms(input_size, False))

n = len(train_ds)
indices = np.random.RandomState(seed).permutation(n).tolist()
n_train = int(n * CONFIG["train_split"])
n_val = int(n * CONFIG["val_split"])
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

bs = CONFIG["batch_size"]
train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(Subset(val_ds, val_idx), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(Subset(val_ds, test_idx), batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {n_train}, Val: {n_val}, Test: {n - n_train - n_val}")

## 8. Visualize Augmented Samples

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for row in range(3):
    idx = train_idx[row * 100]
    # Show original
    orig = val_ds[idx]
    img_orig = orig["image"].permute(1, 2, 0).numpy()
    # Show augmented
    aug = train_ds[idx]
    img_aug = aug["image"].permute(1, 2, 0).numpy()
    mask = orig["mask"].numpy()

    axes[row, 0].imshow(img_orig)
    axes[row, 0].set_title("Original")
    axes[row, 1].imshow(img_aug)
    axes[row, 1].set_title("Augmented (lunar-specific)")
    axes[row, 2].imshow(CLASS_COLORS[mask])
    axes[row, 2].set_title("Ground Truth")

for ax in axes.flat: ax.axis("off")
plt.suptitle("Lunar Augmentation Examples", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 9. Build Model

In [ ]:
if USE_DINOV2:
    model = DINOv2UNet(
        classes=CONFIG["num_classes"],
        frozen_encoder=False,
        dropout_p=CONFIG["mc_dropout_p"],
    ).to(device)
    encoder_name = "DINOv2-Base (ViT-B/14)"
else:
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=CONFIG["num_classes"],
    ).to(device)
    encoder_name = "ResNet-34 (ImageNet)"

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: U-Net + {encoder_name}")
print(f"Trainable params: {params:,}")

criterion = DiceCELoss(CONFIG["dice_weight"], CONFIG["ce_weight"])
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"], eta_min=1e-7)

## 10. Training

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0.0
    for batch in loader:
        imgs, masks = batch["image"].to(device), batch["mask"].to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        total += loss.item() * imgs.size(0)
    return total / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, nc):
    model.eval()
    total, preds, targets = 0.0, [], []
    for batch in loader:
        imgs, masks = batch["image"].to(device), batch["mask"].to(device)
        logits = model(imgs)
        total += criterion(logits, masks).item() * imgs.size(0)
        preds.append(logits.argmax(1).cpu())
        targets.append(masks.cpu())
    p, t = torch.cat(preds), torch.cat(targets)
    return {"loss": total / len(loader.dataset), **iou_score(p, t, nc), **dice_score(p, t, nc)}

In [ ]:
nc = CONFIG["num_classes"]
epochs = CONFIG["epochs"]
cn = CONFIG["class_names"]
best_metric = 0.0
history = {"train_loss": [], "val_loss": [], "val_miou": [], "val_dice": [], "lr": []}

print(f"Training {CONFIG['encoder']} for {epochs} epochs...")
print("-" * 95)

for epoch in range(1, epochs + 1):
    t0 = time.time()
    tl = train_one_epoch(model, train_loader, criterion, optimizer)
    val = evaluate(model, val_loader, criterion, nc)
    scheduler.step()
    lr = optimizer.param_groups[0]["lr"]
    dt = time.time() - t0

    history["train_loss"].append(tl)
    history["val_loss"].append(val["loss"])
    history["val_miou"].append(val["mean_iou"])
    history["val_dice"].append(val["mean_dice"])
    history["lr"].append(lr)

    iou_str = " | ".join(
        f"{cn[i]}: {v:.3f}" if v == v else f"{cn[i]}: N/A"
        for i, v in enumerate(val["per_class_iou"])
    )
    print(f"Epoch {epoch:3d}/{epochs} | train: {tl:.4f} | val: {val['loss']:.4f} | "
          f"mIoU: {val['mean_iou']:.4f} | Dice: {val['mean_dice']:.4f} | lr: {lr:.2e} | {dt:.0f}s")
    print(f"         {iou_str}")

    if val["mean_iou"] > best_metric:
        best_metric = val["mean_iou"]
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_metric": best_metric, "config": CONFIG,
        }, "best_segmenter.pt")
        print(f"         ** New best mIoU: {best_metric:.4f} -- saved **")

print(f"\nDone. Best mIoU: {best_metric:.4f}")

## 11. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train")
axes[0].plot(ep, history["val_loss"], label="Val")
axes[0].set(xlabel="Epoch", ylabel="Loss", title="Loss")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(ep, history["val_miou"], label="mIoU", color="green")
axes[1].plot(ep, history["val_dice"], label="Dice", color="orange")
axes[1].set(xlabel="Epoch", ylabel="Score", title="Validation Metrics")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(ep, history["lr"], color="red")
axes[2].set(xlabel="Epoch", ylabel="LR", title="Learning Rate")
axes[2].grid(True, alpha=0.3)

plt.suptitle(f"Training Progress ({CONFIG['encoder']})", fontsize=14)
plt.tight_layout(); plt.show()

## 12. Test Evaluation

In [ ]:
ckpt = torch.load("best_segmenter.pt", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Best model from epoch {ckpt['epoch']} (mIoU: {ckpt['best_metric']:.4f})")

test = evaluate(model, test_loader, criterion, nc)
print(f"\nTest Loss: {test['loss']:.4f}  |  mIoU: {test['mean_iou']:.4f}  |  Dice: {test['mean_dice']:.4f}")
print("\nPer-class IoU / Dice:")
for i in range(nc):
    iou_v = f"{test['per_class_iou'][i]:.4f}" if test['per_class_iou'][i] == test['per_class_iou'][i] else "N/A"
    dice_v = f"{test['per_class_dice'][i]:.4f}" if test['per_class_dice'][i] == test['per_class_dice'][i] else "N/A"
    print(f"  {cn[i]:15s}: IoU={iou_v}  Dice={dice_v}")

## 13. Predictions + Uncertainty Maps

In [ ]:
@torch.no_grad()
def predict(model, img_tensor):
    model.eval()
    return model(img_tensor.unsqueeze(0).to(device)).argmax(1).squeeze(0).cpu().numpy()


def mc_predict(model, img_tensor, n_samples=20, nc=4):
    """MC Dropout: run n forward passes with dropout on, compute uncertainty."""
    model.eval()
    # Re-enable dropout
    for m in model.modules():
        if isinstance(m, (nn.Dropout, nn.Dropout2d)):
            m.train()

    x = img_tensor.unsqueeze(0).to(device)
    probs = []
    with torch.no_grad():
        for _ in range(n_samples):
            probs.append(F.softmax(model(x), dim=1).cpu())

    probs = torch.stack(probs).squeeze(1)  # (n, C, H, W)
    mean_p = probs.mean(0)  # (C, H, W)
    pred = mean_p.argmax(0).numpy()
    entropy = -(mean_p * torch.log(mean_p + 1e-10)).sum(0).numpy()
    return pred, entropy


fig, axes = plt.subplots(4, 4, figsize=(20, 20))
cols = ["Input", "Ground Truth", "Prediction", "Uncertainty"]
for c, title in enumerate(cols):
    axes[0, c].set_title(title, fontsize=14, fontweight="bold")

has_dropout = USE_DINOV2  # DINOv2UNet has built-in dropout

for row in range(4):
    idx = test_idx[row * 50]
    sample = val_ds[idx]
    img = sample["image"]
    gt = sample["mask"].numpy()

    if has_dropout:
        pred, uncertainty = mc_predict(model, img, n_samples=20, nc=nc)
    else:
        pred = predict(model, img)
        uncertainty = np.zeros_like(gt, dtype=np.float32)

    axes[row, 0].imshow(img.permute(1, 2, 0).numpy())
    axes[row, 1].imshow(CLASS_COLORS[gt])
    axes[row, 2].imshow(CLASS_COLORS[pred])
    axes[row, 3].imshow(uncertainty, cmap="hot")

for ax in axes.flat: ax.axis("off")

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c/255, label=n) for c, n in zip(CLASS_COLORS, cn)]
fig.legend(handles=legend_elements, loc="lower center", ncol=4, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.suptitle(f"Predictions + MC Dropout Uncertainty ({CONFIG['encoder']})", fontsize=16)
plt.tight_layout(); plt.show()

## 14. Sim-to-Real: Real Moon Images

In [ ]:
real_images = sorted(real_moon_dir.glob("*.png"))
print(f"{len(real_images)} real moon images")

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
cols = ["Real Image", "Prediction Overlay", "Uncertainty"]
for c, t in enumerate(cols):
    axes[0, c].set_title(t, fontsize=14, fontweight="bold")

for row in range(4):
    img = cv2.cvtColor(cv2.imread(str(real_images[row * 5])), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    cs = min(h, w, input_size)
    cropped = A.CenterCrop(height=cs, width=cs)(image=img)["image"]
    if cs != input_size:
        cropped = cv2.resize(cropped, (input_size, input_size))

    img_t = torch.from_numpy(cropped.transpose(2, 0, 1).astype(np.float32) / 255.0)

    if has_dropout:
        pred, unc = mc_predict(model, img_t, n_samples=15, nc=nc)
    else:
        pred = predict(model, img_t)
        unc = np.zeros(pred.shape, dtype=np.float32)

    overlay = np.clip(0.6 * cropped / 255.0 + 0.4 * CLASS_COLORS[pred] / 255.0, 0, 1)

    axes[row, 0].imshow(cropped)
    axes[row, 0].set_ylabel(real_images[row * 5].name, fontsize=10)
    axes[row, 1].imshow(overlay)
    axes[row, 2].imshow(unc, cmap="hot")

for ax in axes.flat: ax.axis("off")
fig.legend(handles=legend_elements, loc="lower center", ncol=4, fontsize=12, bbox_to_anchor=(0.5, -0.02))
plt.suptitle("Sim-to-Real Transfer + Uncertainty", fontsize=16)
plt.tight_layout(); plt.show()

## 15. Save & Download

In [ ]:
results = {
    "config": CONFIG,
    "best_epoch": ckpt["epoch"],
    "best_val_miou": round(ckpt["best_metric"], 4),
    "test_loss": round(test["loss"], 5),
    "test_mean_iou": round(test["mean_iou"], 4),
    "test_mean_dice": round(test["mean_dice"], 4),
    "test_per_class_iou": {cn[i]: round(v, 4) if v == v else None for i, v in enumerate(test["per_class_iou"])},
    "test_per_class_dice": {cn[i]: round(v, 4) if v == v else None for i, v in enumerate(test["per_class_dice"])},
    "training_history": history,
}
with open("segmenter_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: best_segmenter.pt, segmenter_results.json")
print("\nCopy to your local project:")
print("  best_segmenter.pt      -> models/best_segmenter.pt")
print("  segmenter_results.json -> outputs/logs/segmenter_results.json")

In [ ]:
try:
    from google.colab import files
    files.download("best_segmenter.pt")
    files.download("segmenter_results.json")
except ImportError:
    print("Not in Colab -- files saved locally.")